# MITS: QLoRA Fine-tuning GLM-4.7-Flash

Fine-tune GLM-4.7-Flash on Socratic tutoring dialogs using Unsloth + TRL.

**Requirements**: Colab Pro+ with A100 GPU

**Pipeline**: Synthetic dialogs → ChatML format → QLoRA training → LoRA export → GGUF conversion → Ollama deploy

In [ ]:
# Install Unsloth (Colab optimized)
!pip install unsloth
!pip install --no-deps trl peft accelerate bitsandbytes

In [ ]:
import torch
print(f"CUDA: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

## 1. Load Model with Unsloth

In [ ]:
from unsloth import FastLanguageModel

# GLM-4.7-Flash: 30B total params, ~3B active (MoE)
MODEL_NAME = "THUDM/glm-4-9b-chat"  # Use 9B chat variant for fine-tuning
MAX_SEQ_LENGTH = 2048
DTYPE = None  # Auto-detect
LOAD_IN_4BIT = True  # QLoRA 4-bit

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)

print(f"Model loaded: {MODEL_NAME}")

## 2. Apply LoRA Adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=32,                 # LoRA rank
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=64,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",  # Unsloth optimized
    random_state=42,
)

# Print trainable params
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

## 3. Load Training Data

In [ ]:
from datasets import load_dataset

# Upload chatml_ready.jsonl to Colab (or mount Google Drive)
# from google.colab import drive
# drive.mount('/content/drive')
# TRAIN_FILE = "/content/drive/MyDrive/MITS/data/training/chatml_ready_train.jsonl"

TRAIN_FILE = "data/training/chatml_ready_train.jsonl"  # Local path

dataset = load_dataset("json", data_files=TRAIN_FILE, split="train")
print(f"Training samples: {len(dataset)}")
print(f"Sample keys: {list(dataset[0].keys())}")

# Preview a sample
sample = dataset[0]
print(f"\nSample text (first 500 chars):")
print(sample["text"][:500])

## 4. Training with TRL SFTTrainer

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        output_dir="outputs/mits-tutor-glm",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=50,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        save_strategy="epoch",
        save_total_limit=3,
    ),
)

In [ ]:
# Check GPU memory before training
gpu_stats = torch.cuda.get_device_properties(0)
used = torch.cuda.memory_allocated() / 1024**3
total = gpu_stats.total_mem / 1024**3
print(f"GPU: {gpu_stats.name}")
print(f"VRAM: {used:.1f} / {total:.1f} GB used")
print(f"Starting training...")

In [ ]:
# Train!
train_result = trainer.train()

print(f"\nTraining complete!")
print(f"Metrics: {train_result.metrics}")

## 5. Save LoRA Adapter

In [ ]:
# Save LoRA adapter
LORA_OUTPUT = "outputs/mits-tutor-glm/lora"
model.save_pretrained(LORA_OUTPUT)
tokenizer.save_pretrained(LORA_OUTPUT)
print(f"LoRA adapter saved to {LORA_OUTPUT}")

# List saved files
import os
for f in os.listdir(LORA_OUTPUT):
    size = os.path.getsize(os.path.join(LORA_OUTPUT, f)) / 1024**2
    print(f"  {f}: {size:.1f} MB")

## 6. Export to GGUF for Ollama

In [ ]:
# Export LoRA to GGUF format
# Method 1: Unsloth built-in export
model.save_pretrained_gguf(
    "outputs/mits-tutor-glm/gguf",
    tokenizer,
    quantization_method="q4_k_m",  # Good balance of quality/size
)
print("GGUF export complete!")

In [ ]:
# Alternative: Export just the LoRA adapter as GGUF
model.save_pretrained_gguf(
    "outputs/mits-tutor-glm/lora_gguf",
    tokenizer,
    quantization_method="f16",  # Keep adapter in f16 for quality
)
print("LoRA GGUF export complete!")

## 7. Test Generation

In [ ]:
# Quick generation test
FastLanguageModel.for_inference(model)

test_prompts = [
    "Не понимаю как найти производную $x^3 + 2x$",
    "Как вычислить $\\int x^2 dx$?",
    "Помогите решить уравнение $x^2 - 5x + 6 = 0$",
]

for prompt in test_prompts:
    messages = [
        {"role": "system", "content": "Ты — сократический репетитор по математике."},
        {"role": "user", "content": prompt},
    ]
    
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    
    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=256,
        temperature=0.7,
        top_p=0.95,
    )
    
    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    print(f"\nQ: {prompt}")
    print(f"A: {response[:200]}")
    print("-" * 60)

## 8. Save to Google Drive

In [ ]:
# Copy outputs to Google Drive for persistence
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r outputs/mits-tutor-glm /content/drive/MyDrive/MITS/outputs/
# print("Saved to Google Drive!")